In [1]:
import pandas as pd

In [2]:
import pandas as pd

strength = pd.read_csv("../data/processed/team_strength_score.csv")

strength.head()


,Rank,Team,Team Strength Score
0,1,Spain,90.770437
1,2,France,89.300662
2,3,Argentina,87.659220
3,4,England,83.006196
4,5,Germany,78.959479


In [3]:
strength.columns

Index(['Rank', 'Team', 'Team Strength Score'], dtype='str')

In [4]:
strength.info()

<class 'pandas.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Rank                 48 non-null     int64  
 1   Team                 48 non-null     str    
 2   Team Strength Score  48 non-null     float64
dtypes: float64(1), int64(1), str(1)
memory usage: 1.3 KB


In [5]:
strength.sort_values(
    by="Team Strength Score",
    ascending=False
).head(10)

,Rank,Team,Team Strength Score
0,1,Spain,90.770437
1,2,France,89.300662
2,3,Argentina,87.659220
3,4,England,83.006196
4,5,Germany,78.959479
5,6,Morocco,77.365753
6,7,Netherlands,75.516276
7,8,Portugal,75.487415
8,9,Brazil,70.915717
9,10,Senegal,67.344860


In [6]:
groups_data = [
    ["A", "Mexico"], ["A", "South Africa"], ["A", "South Korea"], ["A", "Czechia"],
    ["B", "Canada"], ["B", "Switzerland"], ["B", "Qatar"], ["B", "Bosnia and Herzegovina"],
    ["C", "Brazil"], ["C", "Morocco"], ["C", "Scotland"], ["C", "Haiti"],
    ["D", "United States"], ["D", "Paraguay"], ["D", "Australia"], ["D", "Turkiye"],
    ["E", "Germany"], ["E", "Curacao"], ["E", "Cote d'Ivoire"], ["E", "Ecuador"],
    ["F", "Netherlands"], ["F", "Japan"], ["F", "Sweden"], ["F", "Tunisia"],
    ["G", "Belgium"], ["G", "Egypt"], ["G", "Iran"], ["G", "New Zealand"],
    ["H", "Spain"], ["H", "Cape Verde"], ["H", "Saudi Arabia"], ["H", "Uruguay"],
    ["I", "France"], ["I", "Senegal"], ["I", "Iraq"], ["I", "Norway"],
    ["J", "Argentina"], ["J", "Algeria"], ["J", "Austria"], ["J", "Jordan"],
    ["K", "Portugal"], ["K", "DR Congo"], ["K", "Uzbekistan"], ["K", "Colombia"],
    ["L", "England"], ["L", "Croatia"], ["L", "Ghana"], ["L", "Panama"],
]

groups = pd.DataFrame(groups_data, columns=["Group", "Team"])

groups.head(12)

,Group,Team
0,A,Mexico
1,A,South Africa
2,A,South Korea
3,A,Czechia
4,B,Canada
5,B,Switzerland
6,B,Qatar
7,B,Bosnia and Herzegovina
8,C,Brazil
9,C,Morocco


In [7]:
groups.shape

(48, 2)

In [8]:
groups_strength = groups.merge(
    strength,
    on="Team",
    how="left"
)

groups_strength.head()

,Group,Team,Rank,Team Strength Score
0,A,Mexico,14.0,61.513228
1,A,South Africa,31.0,40.900122
2,A,South Korea,19.0,52.580364
3,A,Czechia,NaN,NaN
4,B,Canada,32.0,40.571226


In [9]:
groups_strength[groups_strength["Team Strength Score"].isna()]

,Group,Team,Rank,Team Strength Score
3,A,Czechia,NaN,NaN
15,D,Turkiye,NaN,NaN
17,E,Curacao,NaN,NaN
18,E,Cote d'Ivoire,NaN,NaN


In [10]:
strength[
    strength["Team"].str.contains(
        "Czech|Turkey|Turk|Cura|Ivory|Cote",
        case=False,
        na=False
    )
]


,Rank,Team,Team Strength Score
17,18,Ivory Coast,53.043269
22,23,Turkey,50.321104
25,26,Czech Republic,45.659898
45,46,Curaçao,17.035878


In [11]:
name_corrections = {
    "Czechia": "Czech Republic",
    "Turkiye": "Turkey",
    "Curacao": "Curaçao",
    "Cote d'Ivoire": "Ivory Coast"
}

groups["Team"] = groups["Team"].replace(name_corrections)

In [12]:
groups_strength = groups.merge(
    strength,
    on="Team",
    how="left"
)

In [13]:
groups_strength[groups_strength["Team Strength Score"].isna()]

,Group,Team,Rank,Team Strength Score


In [14]:
group_predictions = groups_strength.sort_values(
    by=["Group", "Team Strength Score"],
    ascending=[True, False]
)

group_predictions[["Group", "Team", "Team Strength Score"]]

,Group,Team,Team Strength Score
0,A,Mexico,61.513228
2,A,South Korea,52.580364
3,A,Czech Republic,45.659898
1,A,South Africa,40.900122
5,B,Switzerland,51.727029
4,B,Canada,40.571226
7,B,Bosnia and Herzegovina,24.524992
6,B,Qatar,10.295073
9,C,Morocco,77.365753
8,C,Brazil,70.915717


In [15]:
group_predictions.to_csv(
    "../data/processed/group_stage_predictions.csv",
    index=False
)

In [16]:
group_predictions["Group Position"] = (
    group_predictions
    .groupby("Group")
    .cumcount() + 1
)

group_predictions[["Group", "Group Position", "Team", "Team Strength Score"]]

,Group,Group Position,Team,Team Strength Score
0,A,1,Mexico,61.513228
2,A,2,South Korea,52.580364
3,A,3,Czech Republic,45.659898
1,A,4,South Africa,40.900122
5,B,1,Switzerland,51.727029
4,B,2,Canada,40.571226
7,B,3,Bosnia and Herzegovina,24.524992
6,B,4,Qatar,10.295073
9,C,1,Morocco,77.365753
8,C,2,Brazil,70.915717


In [17]:
qualified_direct = group_predictions[
    group_predictions["Group Position"] <= 2
]

qualified_direct[["Group", "Group Position", "Team"]]

,Group,Group Position,Team
0,A,1,Mexico
2,A,2,South Korea
5,B,1,Switzerland
4,B,2,Canada
9,C,1,Morocco
8,C,2,Brazil
15,D,1,Turkey
12,D,2,United States
16,E,1,Germany
19,E,2,Ecuador


In [18]:
len(qualified_direct)

24

In [19]:
third_place_teams = group_predictions[
    group_predictions["Group Position"] == 3
]

best_thirds = third_place_teams.sort_values(
    by="Team Strength Score",
    ascending=False
).head(8)

best_thirds[["Group", "Group Position", "Team", "Team Strength Score"]]

,Group,Group Position,Team,Team Strength Score
18,E,3,Ivory Coast,53.043269
35,I,3,Norway,50.026573
22,F,3,Sweden,45.876301
3,A,3,Czech Republic,45.659898
14,D,3,Australia,43.738805
38,J,3,Austria,43.063526
41,K,3,DR Congo,37.013713
47,L,3,Panama,36.559617


In [20]:
len(best_thirds)

8

In [21]:
round_of_32 = pd.concat(
    [qualified_direct, best_thirds],
    ignore_index=True
)

len(round_of_32)

32

In [22]:
round_of_32[["Team", "Team Strength Score"]].sort_values(
    by="Team Strength Score",
    ascending=False
)

,Team,Team Strength Score
14,Spain,90.770437
16,France,89.300662
18,Argentina,87.659220
22,England,83.006196
8,Germany,78.959479
4,Morocco,77.365753
10,Netherlands,75.516276
20,Portugal,75.487415
5,Brazil,70.915717
17,Senegal,67.344860


In [23]:
round32_matches = [
    [73, "A2", "B2"],
    [75, "F1", "C2"],
    [76, "C1", "F2"],
    [78, "E2", "I2"],
    [83, "K2", "L2"],
    [84, "H1", "J2"],
    [86, "J1", "H2"],
    [88, "D2", "G2"],
]

round32 = pd.DataFrame(
    round32_matches,
    columns=["Match", "Slot 1", "Slot 2"]
)

round32

,Match,Slot 1,Slot 2
0,73,A2,B2
1,75,F1,C2
2,76,C1,F2
3,78,E2,I2
4,83,K2,L2
5,84,H1,J2
6,86,J1,H2
7,88,D2,G2


In [24]:
group_predictions["Slot"] = (
    group_predictions["Group"] +
    group_predictions["Group Position"].astype(str)
)

group_predictions[["Slot", "Team", "Team Strength Score"]].head(12)

,Slot,Team,Team Strength Score
0,A1,Mexico,61.513228
2,A2,South Korea,52.580364
3,A3,Czech Republic,45.659898
1,A4,South Africa,40.900122
5,B1,Switzerland,51.727029
4,B2,Canada,40.571226
7,B3,Bosnia and Herzegovina,24.524992
6,B4,Qatar,10.295073
9,C1,Morocco,77.365753
8,C2,Brazil,70.915717


In [25]:
slot_to_team = group_predictions.set_index("Slot")["Team"].to_dict()
slot_to_score = group_predictions.set_index("Slot")["Team Strength Score"].to_dict()

In [26]:
slot_to_team["A2"]

'South Korea'

In [27]:
round32["Team 1"] = round32["Slot 1"].map(slot_to_team)
round32["Team 2"] = round32["Slot 2"].map(slot_to_team)

round32

,Match,Slot 1,Slot 2,Team 1,Team 2
0,73,A2,B2,South Korea,Canada
1,75,F1,C2,Netherlands,Brazil
2,76,C1,F2,Morocco,Japan
3,78,E2,I2,Ecuador,Senegal
4,83,K2,L2,Colombia,Croatia
5,84,H1,J2,Spain,Algeria
6,86,J1,H2,Argentina,Uruguay
7,88,D2,G2,United States,Iran


In [28]:
round32["Score 1"] = round32["Slot 1"].map(slot_to_score)
round32["Score 2"] = round32["Slot 2"].map(slot_to_score)

round32

,Match,Slot 1,Slot 2,Team 1,Team 2,Score 1,Score 2
0,73,A2,B2,South Korea,Canada,52.580364,40.571226
1,75,F1,C2,Netherlands,Brazil,75.516276,70.915717
2,76,C1,F2,Morocco,Japan,77.365753,64.358465
3,78,E2,I2,Ecuador,Senegal,54.648724,67.344860
4,83,K2,L2,Colombia,Croatia,58.541659,62.782893
5,84,H1,J2,Spain,Algeria,90.770437,55.740735
6,86,J1,H2,Argentina,Uruguay,87.659220,51.035357
7,88,D2,G2,United States,Iran,44.252178,51.984841


In [29]:
best_thirds["Slot"] = (
    best_thirds["Group"] +
    best_thirds["Group Position"].astype(str)
)

best_thirds[["Slot", "Team"]]

,Slot,Team
18,E3,Ivory Coast
35,I3,Norway
22,F3,Sweden
3,A3,Czech Republic
14,D3,Australia
38,J3,Austria
41,K3,DR Congo
47,L3,Panama


In [30]:
third_place_matches = [
    [74, "E1", "E3"],
    [77, "I1", "D3"],
    [79, "A1", "F3"],
    [80, "L1", "J3"],
    [81, "D1", "I3"],
    [82, "G1", "A3"],
    [85, "B1", "E3"],
    [87, "K1", "L3"],
]

In [31]:
third_place_matches = [
    [74, "E1", "F3"],
    [77, "I1", "D3"],
    [79, "A1", "E3"],
    [80, "L1", "J3"],
    [81, "D1", "I3"],
    [82, "G1", "A3"],
    [85, "B1", "K3"],
    [87, "K1", "L3"],
]

In [32]:
third_round32 = pd.DataFrame(
    third_place_matches,
    columns=["Match", "Slot 1", "Slot 2"]
)

third_round32

,Match,Slot 1,Slot 2
0,74,E1,F3
1,77,I1,D3
2,79,A1,E3
3,80,L1,J3
4,81,D1,I3
5,82,G1,A3
6,85,B1,K3
7,87,K1,L3


In [33]:
third_round32["Team 1"] = third_round32["Slot 1"].map(slot_to_team)
third_round32["Team 2"] = third_round32["Slot 2"].map(slot_to_team)

third_round32["Score 1"] = third_round32["Slot 1"].map(slot_to_score)
third_round32["Score 2"] = third_round32["Slot 2"].map(slot_to_score)

third_round32

,Match,Slot 1,Slot 2,Team 1,Team 2,Score 1,Score 2
0,74,E1,F3,Germany,Sweden,78.959479,45.876301
1,77,I1,D3,France,Australia,89.300662,43.738805
2,79,A1,E3,Mexico,Ivory Coast,61.513228,53.043269
3,80,L1,J3,England,Austria,83.006196,43.063526
4,81,D1,I3,Turkey,Norway,50.321104,50.026573
5,82,G1,A3,Belgium,Czech Republic,67.326950,45.659898
6,85,B1,K3,Switzerland,DR Congo,51.727029,37.013713
7,87,K1,L3,Portugal,Panama,75.487415,36.559617


In [34]:
third_round32["Winner"] = third_round32.apply(
    lambda row: row["Team 1"]
    if row["Score 1"] > row["Score 2"]
    else row["Team 2"],
    axis=1
)

third_round32[["Match", "Team 1", "Team 2", "Winner"]]

,Match,Team 1,Team 2,Winner
0,74,Germany,Sweden,Germany
1,77,France,Australia,France
2,79,Mexico,Ivory Coast,Mexico
3,80,England,Austria,England
4,81,Turkey,Norway,Turkey
5,82,Belgium,Czech Republic,Belgium
6,85,Switzerland,DR Congo,Switzerland
7,87,Portugal,Panama,Portugal


In [35]:
round32_complete = pd.concat(
    [round32, third_round32],
    ignore_index=True
).sort_values("Match")

round32_complete[["Match", "Team 1", "Team 2", "Winner"]]

,Match,Team 1,Team 2,Winner
0,73,South Korea,Canada,NaN
8,74,Germany,Sweden,Germany
1,75,Netherlands,Brazil,NaN
2,76,Morocco,Japan,NaN
9,77,France,Australia,France
3,78,Ecuador,Senegal,NaN
10,79,Mexico,Ivory Coast,Mexico
11,80,England,Austria,England
12,81,Turkey,Norway,Turkey
13,82,Belgium,Czech Republic,Belgium


In [36]:
round32_complete.to_csv(
    "../data/processed/round_of_32_predictions.csv",
    index=False
)

In [37]:
len(round32_complete)

16

In [38]:
round32_complete["Winner"] = round32_complete.apply(
    lambda row: row["Team 1"]
    if row["Score 1"] > row["Score 2"]
    else row["Team 2"],
    axis=1
)

round32_complete[["Match", "Team 1", "Team 2", "Winner"]]

,Match,Team 1,Team 2,Winner
0,73,South Korea,Canada,South Korea
8,74,Germany,Sweden,Germany
1,75,Netherlands,Brazil,Netherlands
2,76,Morocco,Japan,Morocco
9,77,France,Australia,France
3,78,Ecuador,Senegal,Senegal
10,79,Mexico,Ivory Coast,Mexico
11,80,England,Austria,England
12,81,Turkey,Norway,Turkey
13,82,Belgium,Czech Republic,Belgium


In [39]:
round32_complete["Winner"].isna().sum()

np.int64(0)

In [40]:
round32_complete[["Match", "Winner"]]

,Match,Winner
0,73,South Korea
8,74,Germany
1,75,Netherlands
2,76,Morocco
9,77,France
3,78,Senegal
10,79,Mexico
11,80,England
12,81,Turkey
13,82,Belgium


In [41]:
round16_matches = [
    [89, "South Korea", "Netherlands"],
    [90, "Germany", "France"],

    [91, "Morocco", "Senegal"],
    [92, "Mexico", "England"],

    [93, "Croatia", "Spain"],
    [94, "Turkey", "Belgium"],

    [95, "Argentina", "Iran"],
    [96, "Switzerland", "Portugal"]
]

round16 = pd.DataFrame(
    round16_matches,
    columns=["Match", "Team 1", "Team 2"]
)

round16

,Match,Team 1,Team 2
0,89,South Korea,Netherlands
1,90,Germany,France
2,91,Morocco,Senegal
3,92,Mexico,England
4,93,Croatia,Spain
5,94,Turkey,Belgium
6,95,Argentina,Iran
7,96,Switzerland,Portugal


In [42]:
round16_matches = [
    [89, "Germany", "France"],
    [90, "South Korea", "Netherlands"],

    [91, "Morocco", "Senegal"],
    [92, "Mexico", "England"],

    [93, "Croatia", "Spain"],
    [94, "Turkey", "Belgium"],

    [95, "Argentina", "Iran"],
    [96, "Switzerland", "Portugal"]
]

In [43]:
round16

,Match,Team 1,Team 2
0,89,South Korea,Netherlands
1,90,Germany,France
2,91,Morocco,Senegal
3,92,Mexico,England
4,93,Croatia,Spain
5,94,Turkey,Belgium
6,95,Argentina,Iran
7,96,Switzerland,Portugal


In [44]:
round16 = pd.DataFrame(
    round16_matches,
    columns=["Match", "Team 1", "Team 2"]
)

round16

,Match,Team 1,Team 2
0,89,Germany,France
1,90,South Korea,Netherlands
2,91,Morocco,Senegal
3,92,Mexico,England
4,93,Croatia,Spain
5,94,Turkey,Belgium
6,95,Argentina,Iran
7,96,Switzerland,Portugal


In [45]:
team_to_score = strength.set_index("Team")["Team Strength Score"].to_dict()

round16["Score 1"] = round16["Team 1"].map(team_to_score)
round16["Score 2"] = round16["Team 2"].map(team_to_score)

In [46]:
round16

,Match,Team 1,Team 2,Score 1,Score 2
0,89,Germany,France,78.959479,89.300662
1,90,South Korea,Netherlands,52.580364,75.516276
2,91,Morocco,Senegal,77.365753,67.344860
3,92,Mexico,England,61.513228,83.006196
4,93,Croatia,Spain,62.782893,90.770437
5,94,Turkey,Belgium,50.321104,67.326950
6,95,Argentina,Iran,87.659220,51.984841
7,96,Switzerland,Portugal,51.727029,75.487415


In [47]:
round16["Winner"] = round16.apply(
    lambda row: row["Team 1"]
    if row["Score 1"] > row["Score 2"]
    else row["Team 2"],
    axis=1
)

round16[["Match", "Team 1", "Team 2", "Winner"]]

,Match,Team 1,Team 2,Winner
0,89,Germany,France,France
1,90,South Korea,Netherlands,Netherlands
2,91,Morocco,Senegal,Morocco
3,92,Mexico,England,England
4,93,Croatia,Spain,Spain
5,94,Turkey,Belgium,Belgium
6,95,Argentina,Iran,Argentina
7,96,Switzerland,Portugal,Portugal


In [48]:
quarterfinals = pd.DataFrame([
    [97, "France", "Netherlands"],
    [98, "Spain", "Belgium"],
    [99, "Morocco", "England"],
    [100, "Argentina", "Portugal"]
], columns=["Match", "Team 1", "Team 2"])

quarterfinals

,Match,Team 1,Team 2
0,97,France,Netherlands
1,98,Spain,Belgium
2,99,Morocco,England
3,100,Argentina,Portugal


In [49]:
quarterfinals["Score 1"] = quarterfinals["Team 1"].map(team_to_score)
quarterfinals["Score 2"] = quarterfinals["Team 2"].map(team_to_score)

quarterfinals

,Match,Team 1,Team 2,Score 1,Score 2
0,97,France,Netherlands,89.300662,75.516276
1,98,Spain,Belgium,90.770437,67.326950
2,99,Morocco,England,77.365753,83.006196
3,100,Argentina,Portugal,87.659220,75.487415


In [50]:
quarterfinals["Winner"] = quarterfinals.apply(
    lambda row: row["Team 1"]
    if row["Score 1"] > row["Score 2"]
    else row["Team 2"],
    axis=1
)

quarterfinals[["Match", "Team 1", "Team 2", "Winner"]]

,Match,Team 1,Team 2,Winner
0,97,France,Netherlands,France
1,98,Spain,Belgium,Spain
2,99,Morocco,England,England
3,100,Argentina,Portugal,Argentina


In [51]:
semifinals = pd.DataFrame([
    [101, "France", "Spain"],
    [102, "England", "Argentina"]
], columns=["Match", "Team 1", "Team 2"])

semifinals

,Match,Team 1,Team 2
0,101,France,Spain
1,102,England,Argentina


In [52]:
semifinals["Score 1"] = semifinals["Team 1"].map(team_to_score)
semifinals["Score 2"] = semifinals["Team 2"].map(team_to_score)

semifinals

,Match,Team 1,Team 2,Score 1,Score 2
0,101,France,Spain,89.300662,90.770437
1,102,England,Argentina,83.006196,87.659220


In [53]:
semifinals["Winner"] = semifinals.apply(
    lambda row: row["Team 1"]
    if row["Score 1"] > row["Score 2"]
    else row["Team 2"],
    axis=1
)

semifinals[["Match", "Team 1", "Team 2", "Winner"]]

,Match,Team 1,Team 2,Winner
0,101,France,Spain,Spain
1,102,England,Argentina,Argentina


In [54]:
final = pd.DataFrame([
    [103, "Spain", "Argentina"]
], columns=["Match", "Team 1", "Team 2"])

final

,Match,Team 1,Team 2
0,103,Spain,Argentina


In [55]:
final["Score 1"] = final["Team 1"].map(team_to_score)
final["Score 2"] = final["Team 2"].map(team_to_score)

final

,Match,Team 1,Team 2,Score 1,Score 2
0,103,Spain,Argentina,90.770437,87.65922


In [56]:
final["Winner"] = final.apply(
    lambda row: row["Team 1"]
    if row["Score 1"] > row["Score 2"]
    else row["Team 2"],
    axis=1
)

final[["Team 1", "Team 2", "Winner"]]

,Team 1,Team 2,Winner
0,Spain,Argentina,Spain


In [57]:
group_stage_predictions.csv
round_of_32_predictions.csv
round_of_16_predictions.csv
quarterfinal_predictions.csv
semifinal_predictions.csv
final_prediction.csv

NameError: name 'group_stage_predictions' is not defined

In [58]:
round32_complete.to_csv(
    "../data/processed/round_of_32_predictions.csv",
    index=False
)

round16.to_csv(
    "../data/processed/round_of_16_predictions.csv",
    index=False
)

quarterfinals.to_csv(
    "../data/processed/quarterfinal_predictions.csv",
    index=False
)

semifinals.to_csv(
    "../data/processed/semifinal_predictions.csv",
    index=False
)

final.to_csv(
    "../data/processed/final_prediction.csv",
    index=False
)

In [59]:
import os

os.listdir("../data/processed")

['final_prediction.csv',
 'group_stage_predictions.csv',
 'quarterfinal_predictions.csv',
 'round_of_16_predictions.csv',
 'round_of_32_predictions.csv',
 'semifinal_predictions.csv',
 'team_strength_full.csv',
 'team_strength_score.csv']